# Week 6: Unsupervised Learning & Clustering
## Notebook 10: Kernel PCA & Non-Linear Manifold Projection

### 1. Conceptual Intuition & Theoretical Background
Standard Principal Component Analysis (PCA) performs orthogonal linear projections:
$$\max_w w^T \Sigma w, \quad \text{s.t. } \|w\|=1$$
However, when data lies on non-linear manifolds (e.g., Swiss roll or concentric rings), linear PCA cannot "unroll" the manifold.

**Kernel PCA (Schölkopf et al., 1998):**
Maps input points into a high-dimensional reproducing kernel Hilbert space (RKHS) via non-linear mapping $\Phi(x)$:
$$K_{ij} = k(x_i, x_j) = \langle \Phi(x_i), \Phi(x_j) \rangle$$
Using the **Radial Basis Function (RBF) Kernel**:
$$k(x_i, x_j) = \exp\left( -\gamma \|x_i - x_j\|^2 \right)$$

**Centering in Feature Space:**
Since $\Phi(x)$ is implicit, the Gram matrix $K$ is centered directly via:
$$\tilde{K} = K - 1_N K - K 1_N + 1_N K 1_N, \quad 1_N = \frac{1}{N} J_N$$

Eigenvalue decomposition of $\tilde{K} v = \lambda v$ yields principal coordinates $\sqrt{\lambda} v$.

---

### 2. Scratch Python Implementation: Kernel PCA

In [ ]:
import numpy as np

class KernelPCAScratch:
    def __init__(self, n_components=2, gamma=15.0):
        self.n_components = n_components
        self.gamma = gamma
        self.alphas = None
        self.lambdas = None
        self.K_fit = None
        self.X_fit = None
        
    def _rbf_kernel(self, X1, X2):
        diff = X1[:, np.newaxis, :] - X2[np.newaxis, :, :]
        dist_sq = np.sum(diff ** 2, axis=-1)
        return np.exp(-self.gamma * dist_sq)
        
    def fit_transform(self, X):
        self.X_fit = X
        N = len(X)
        
        # 1. Compute Pairwise Gram Matrix K
        K = self._rbf_kernel(X, X)
        
        # 2. Center Kernel Matrix in Hilbert Space
        one_n = np.ones((N, N)) / N
        K_tilde = K - np.dot(one_n, K) - np.dot(K, one_n) + np.dot(one_n, np.dot(K, one_n))
        
        # 3. Eigendecomposition of Centered Gram Matrix
        eigenvalues, eigenvectors = np.linalg.eigh(K_tilde)
        
        # Sort descending
        idx = np.argsort(eigenvalues)[::-1]
        self.lambdas = eigenvalues[idx[:self.n_components]]
        self.alphas = eigenvectors[:, idx[:self.n_components]]
        
        # Normalize eigenvectors in feature space: v_k / sqrt(lambda_k)
        # Projected coordinates are alphas * sqrt(lambdas)
        projected = self.alphas * np.sqrt(np.maximum(self.lambdas, 1e-9))
        return projected

# Generate Concentric Circles (Linearly Inseparable)
np.random.seed(42)
n_pts = 100
r_in = np.random.uniform(0.1, 0.4, n_pts)
theta_in = np.random.uniform(0, 2*np.pi, n_pts)
circle_in = np.column_stack([r_in * np.cos(theta_in), r_in * np.sin(theta_in)])

r_out = np.random.uniform(1.2, 1.5, n_pts)
theta_out = np.random.uniform(0, 2*np.pi, n_pts)
circle_out = np.column_stack([r_out * np.cos(theta_out), r_out * np.sin(theta_out)])

X_rings = np.vstack([circle_in, circle_out])
labels = np.array([0]*n_pts + [1]*n_pts)

kpca = KernelPCAScratch(n_components=2, gamma=4.0)
X_projected = kpca.fit_transform(X_rings)

# Measure linear separability on 1st principal component
pc1_class0 = X_projected[labels == 0, 0]
pc1_class1 = X_projected[labels == 1, 0]
overlap = max(pc1_class0.min(), pc1_class1.min()) < min(pc1_class0.max(), pc1_class1.max())

print("--- Kernel PCA Non-Linear Manifold Unrolling ---")
print(f"Original Input Shape:        {X_rings.shape}")
print(f"Kernel Projected Shape:      {X_projected.shape}")
print(f"Class 0 PC1 Bounds:          [{pc1_class0.min():.3f}, {pc1_class0.max():.3f}]")
print(f"Class 1 PC1 Bounds:          [{pc1_class1.min():.3f}, {pc1_class1.max():.3f}]")
print(f"PC1 Linearly Disentangled:   {not overlap}")

### 3. Step-by-Step Code Walkthrough
1. **Gram Matrix ($K$):** Computes all-pairs non-linear affinities without explicitly projecting into infinite-dimensional Hilbert space.
2. **Double Centering:** Subtracts mean row/column projections to preserve the zero-mean covariance assumption.
3. **Linear Disentanglement:** Transforms concentric rings into linearly separable intervals on the first eigenvector.

---

### 4. Industry Applications & Trade-offs
- **Process Fault Detection:** Monitoring non-linear chemical refinery sensor telemetries.
- **Trade-off:** $O(N^2)$ memory and $O(N^3)$ compute scaling limits raw KPCA to datasets with $<50,000$ points without Nyström sampling.

---

### 5. Top Interview Questions & Answers
**Q1: Why is centering in feature space different from centering input data in standard PCA?**
> **Answer:** In standard PCA, centering simply subtracts feature means $\bar{x}$. In Kernel PCA, $\Phi(x)$ is never explicitly evaluated; centering must be carried out algebraically on the Gram matrix $K$ using the projection operator $(I - 1_N) K (I - 1_N)$.

**Q2: What is the "Pre-image Problem" in Kernel PCA?**
> **Answer:** Projecting high-dimensional points back to input space ($z \to x$) is ill-posed because an arbitrary vector in Hilbert space may not correspond to any valid physical point in the original feature domain.